### This notebook is a test of "of_bpdrr.py" as a module that contains the functions of all 6 objective functions of the Battle of Postdisaster Response and Restoration

In [1]:
# Import libraries to be used, including "of_bpdrr", which contains the objective fuctions of the BPDRR
import of_bpdrr
import wntr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Import the water network model
ds_sel = 'DS2' # which damage scenario to analyse
inp_file = "BBM-EPS_"+ds_sel+"mcg.inp"
#inp_file = "BBM-EPS_DS2controls.inp"
wn = wntr.network.WaterNetworkModel(inp_file)

In [3]:
# Simulation settings
wn.options.time.duration = 3 * 24 * 3600       # 3 days
wn.options.time.hydraulic_timestep = 30*60     # 30 minutes
wn.options.time.report_timestep = 30*60        # 30 minutes

wn.options.hydraulic.demand_model = "PDA"

wn.options.hydraulic.required_pressure = 20      # m
wn.options.hydraulic.minimum_pressure = 0
wn.options.hydraulic.pressure_exponent = 0.5

In [4]:
# timestep
dt = wn.options.time.hydraulic_timestep

# timestep in minutes
dtm = dt/60

In [5]:
# run the simulation
sim = wntr.sim.EpanetSimulator(wn)
results = sim.run_sim()

In [6]:
# getting the actual demand at each node over time 
demand = results.node["demand"]*1000  # convert to L/s

In [7]:
# Find all demand nodes (base demand > 0)
demand_nodes = []

# Expected demand at each node
expected = {}

for node_name in wn.junction_name_list:

    node = wn.get_node(node_name)

    if node.base_demand > 0:

        demand_nodes.append(node_name)
        expected[node_name] = node.base_demand*1000  #L/s

print(f"Demand nodes found: {len(demand_nodes)}")

Demand nodes found: 4201


In [8]:
# Finding emitter nodes (nodes starting with "E_")
emitter_nodes = [
    j for j in wn.junction_name_list
    if j.startswith("E_")
]

# Demand at emitter nodes (leakage) over time
leakage = demand[emitter_nodes]

### Functionality 
The percentage of the demand supplied by the distribution system.

In [9]:
functionality_series = of_bpdrr.system_functionality(demand, demand_nodes, expected)

print(functionality_series)

0         44.404114
1800      44.404179
3600      34.221523
5400      34.221611
7200      31.675798
            ...    
252000    91.711067
253800    91.710953
255600    66.465790
257400    66.465881
259200    44.403522
Length: 145, dtype: float32


### 1. OF: Time without supply for hospital/firefighting (FH)
The time that the hospitals and the firefighting flows are without supply It's calculated by multiplying the simulation time step duration  with the number of time steps in which the supply/demand ratio for the hospitals and firefighting flows was less than 0.5.

In [10]:
FH_value, FH_undersupplied = of_bpdrr.OF_FH(wn, demand, dtm,)

print(f"FH = {FH_value:,.2f} minutes") 

FH = 4,260.00 minutes


### 2. rapidity of recovery (t95)
The time that the hospitals and the firefighting flows are without supply It's calculated by multiplying the simulation time step duration  with the number of time steps in which the supply/demand ratio for the hospitals and firefighting flows was less than 0.5.

In [11]:
t95 = of_bpdrr.OF_t95(demand, demand_nodes, expected)

print(f"Rapidity of recovery (t95): {t95:.2f}")

Rapidity of recovery (t95): nan


### 3. OF: Resilience loss (RL)
RL considers the Accumulated loss of functionality during the recovery process of the system. It represents the area between the full functionality line (100%) and the functionality time series

In [12]:
RL = of_bpdrr.OF_RL(demand, demand_nodes, expected, dtm)

print(f"Resilience Loss = {RL:,.2f} %-minutes")

Resilience Loss = 27,652.40 %-minutes


### 4. OF: Average time of no user service (Time no serv.)
Measures the average time each demand node across the network stayed without service. The formula of this objective is similar to the FH formula, but it considers the sum of all demand nodes that were undersupplied (supply/demand ratio under 0.5) divided by the total amount of demand nodes (DN)

In [13]:
time_no_serv = of_bpdrr.OF_time_no_serv(demand, demand_nodes, expected, dtm)

print(f"Average Time of No User Service = {time_no_serv:.2f} minutes")

Average Time of No User Service = 942.09 minutes


### 5. OF: Number of users without service for eight consecutive hours (NWSECH)
Number of demand nodes that stayed without service for more than eight consecutive hours. It’s calculated by counting the amount of nodes with at least one continuous 8-hour period during which the node's supply/demand ratio never exceeds 0.5.

In [14]:
NNS = of_bpdrr.OF_NWSECH(demand, demand_nodes, expected, dtm)

print(f"Nodes without service for 8 consecutive hours = {NNS}")

Nodes without service for 8 consecutive hours = 12


### 6. OF: Water loss (WL)
counts the volume in litres of water lost during the period after the earthquake, by multiplying the time step with the sum of the outflows across all damages in the system.

In [15]:
WL = of_bpdrr.OF_WL(demand, emitter_nodes, dt)

print(f"Water loss = {WL:,.2f} m³")

Water loss = 76,426.97 m³
